In [1]:
import re, json
import numpy as np
import pandas as pd
import scanpy as sc

import matplotlib.pyplot as plt
import seaborn as sns
plt.rcParams['figure.figsize'] = [5, 5]
sc.settings.verbosity = 3
sc.logging.print_header()
sc.set_figure_params(dpi=72)

Download NormanWeissman2019_filtered.h5ad from scPerturb database ([scPerturb Zenodo](https://zenodo.org/records/13350497)). Preprocess this dataset according to the strategy of CPA and GEARS.

## Loading the raw data, rename the old gene names, and filter genes not in prior knowledge

In [ ]:
radata = sc.read("raw/NormanWeissman2019_filtered.h5ad")
radata

AnnData object with n_obs × n_vars = 111445 × 33694
    obs: 'guide_id', 'read_count', 'UMI_count', 'coverage', 'gemgroup', 'good_coverage', 'number_of_cells', 'tissue_type', 'cell_line', 'cancer', 'disease', 'perturbation_type', 'celltype', 'organism', 'perturbation', 'nperts', 'ngenes', 'ncounts', 'percent_mito', 'percent_ribo'
    var: 'ensemble_id', 'ncounts', 'ncells'

Check all control guide

In [3]:
list_control = []
for i in np.unique(radata.obs["guide_id"]):
   m = re.match(r"NegCtrl(.*)_NegCtrl(.*)+NegCtrl(.*)_NegCtrl(.*)", i)
   if m :
    list_control.append(m.group())

list_control

['NegCtrl0_NegCtrl0;NegCtrl0_NegCtrl0',
 'NegCtrl10_NegCtrl0;NegCtrl10_NegCtrl0',
 'NegCtrl11_NegCtrl0;NegCtrl11_NegCtrl0',
 'NegCtrl1_NegCtrl0;NegCtrl1_NegCtrl0']

remove "NegCtrl1_NegCtrl0__NegCtrl1_NegCtrl0" suggested by authors

In [4]:
padata = radata[(radata.obs["guide_id"] != "NegCtrl1_NegCtrl0;NegCtrl1_NegCtrl0") & radata.obs["good_coverage"]].copy()
padata

AnnData object with n_obs × n_vars = 101719 × 33694
    obs: 'guide_id', 'read_count', 'UMI_count', 'coverage', 'gemgroup', 'good_coverage', 'number_of_cells', 'tissue_type', 'cell_line', 'cancer', 'disease', 'perturbation_type', 'celltype', 'organism', 'perturbation', 'nperts', 'ngenes', 'ncounts', 'percent_mito', 'percent_ribo'
    var: 'ensemble_id', 'ncounts', 'ncells'

Load the vocab used in prior network and filter the gene in this dataset

In [ ]:
perturbed_renaming = pd.Series({
    "C3orf72": "FOXL2NB", # C3orf72 is a previous name of FOXL2NB
    "C19orf26": "CBARP", # C19orf26 is a previous name of CBARP
    "KIAA1804": "MAP3K21", # KIAA1804 is a alias name of MAP3K21
    "ELMSAN1":  "MIDEAS", # ELMSAN1 is a previous name of MIDEAS
    "RP5-862P8.2": "MAP3K21", # RP5-862P8.2 is a alias name of MAP3K21 # both names are not in the Vocab
}) # used to rename the perturbed genes and measured genes to the names in the Vocab

In [ ]:
padata.var_names = padata.var_names.map(lambda x: perturbed_renaming.get(x, x))
for old, new in perturbed_renaming.items():
    padata.obs['perturbation'] = padata.obs['perturbation'].map(lambda x: x.replace(old, new))
padata

AnnData object with n_obs × n_vars = 101719 × 33694
    obs: 'guide_id', 'read_count', 'UMI_count', 'coverage', 'gemgroup', 'good_coverage', 'number_of_cells', 'tissue_type', 'cell_line', 'cancer', 'disease', 'perturbation_type', 'celltype', 'organism', 'perturbation', 'nperts', 'ngenes', 'ncounts', 'percent_mito', 'percent_ribo'
    var: 'ensemble_id', 'ncounts', 'ncells'

In [7]:
padata.obs['perturbation'].value_counts().sort_index(ascending=False)

perturbation
control           8395
ZNF318_FOXL2       233
ZNF318             622
ZC3HAV1_HOXC13     590
ZC3HAV1_CEBPE      393
                  ... 
ARRDC3             455
ARID1A             211
AHR_KLF1           451
AHR_FEV            257
AHR                518
Name: count, Length: 237, dtype: int64

## Preprocessing

Keep the count data in a counts layer

In [8]:
padata.X = padata.X.astype(np.int32)
sc.pp.filter_cells(padata, min_counts=3500)
sc.pp.filter_cells(padata, min_genes=200)
sc.pp.filter_genes(padata, min_cells=50)

filtered out 16435 genes that are detected in less than 50 cells


In [9]:
selected_pert_df = padata.obs.groupby(
    ['perturbation', 'cell_line'], observed=True
).size().reset_index(name='count')
selected_pert_df['count'].min()

51

Normalization and HVG selection. It is worth noting that the dataset is extremely unbalanced, which may effect the selected HVGs

In [10]:
padata.layers["raw"] = padata.X.copy()
sc.pp.normalize_total(padata, target_sum=1e4)
sc.pp.log1p(padata)
sc.pp.highly_variable_genes(
    padata, n_top_genes=5000, 
    subset=False, flavor='seurat'
)
padata

normalizing counts per cell
    finished (0:00:02)
extracting highly variable genes
    finished (0:00:05)
--> added
    'highly_variable', boolean vector (adata.var)
    'means', float vector (adata.var)
    'dispersions', float vector (adata.var)
    'dispersions_norm', float vector (adata.var)


AnnData object with n_obs × n_vars = 101719 × 17259
    obs: 'guide_id', 'read_count', 'UMI_count', 'coverage', 'gemgroup', 'good_coverage', 'number_of_cells', 'tissue_type', 'cell_line', 'cancer', 'disease', 'perturbation_type', 'celltype', 'organism', 'perturbation', 'nperts', 'ngenes', 'ncounts', 'percent_mito', 'percent_ribo', 'n_counts', 'n_genes'
    var: 'ensemble_id', 'ncounts', 'ncells', 'n_cells', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'log1p', 'hvg'
    layers: 'raw'

In [14]:
padata.write_h5ad('preprocessed.h5ad')

## split dataset

In [ ]:
adata = sc.read("preprocessed.h5ad", backed='r')
split_df = adata.obs[['perturbation']].reset_index(names='cell')
np.random.seed(42)
split_df['split'] = np.random.choice(['train', 'val', 'test'], size=len(split_df), p=[0.6, 0.1, 0.3])
split_df['subsplit'] = split_df.apply(
    lambda x: 'combo' if '_' in x['perturbation'] else 'single',
    axis=1
)
split_df['split'] = split_df.apply(
    lambda x: 'test' if x['subsplit'] == 'combo' and x['split'] == 'train' else x['split'],
    axis=1
)
selected_ctrl = split_df.query('subsplit == "single" and perturbation == "control"').sample(frac=2/8, random_state=42).index.values
selected_ctrl4val = selected_ctrl[:len(selected_ctrl)//2]
selected_ctrl4test = selected_ctrl[len(selected_ctrl)//2:]
split_df.loc[selected_ctrl4val, ['split', 'subsplit']] = np.array(['val', 'combo']).reshape(1, 2)
split_df.loc[selected_ctrl4test, ['split', 'subsplit']] = np.array(['test', 'combo']).reshape(1, 2)

split  subsplit
train  single      36350
test   combo       36213
       single      17993
val    single       6129
       combo        5034
Name: count, dtype: int64

In [ ]:
split_df.set_index('cell').to_parquet('split_single_seencombo2better.parquet')